# 09b — Study Area Overview (world map + per-city OSM panels)

One figure, two rows, following the `paper-figure-style` house
convention (final-size canvas, trimmed spines, direct labelling, panel
letters):

- **Row 1** -- a single world map with all four study-area cities
  pinpointed, each encoded by both colour and marker shape (redundant
  coding, so it survives greyscale/CVD), with a legend naming them.
- **Row 2** -- one panel per city, side by side (`City A | City B |
  City C | City D`), each on a real OpenStreetMap basemap (Cartopy's
  `OSM` tile source), showing that city's actual study-area boundary
  and its positive (crash) / negative (generated) points, with
  longitude/latitude tick labels on both axes.

**Read-only.** Reads only `paths.yaml`'s per-city `boundary_geojson`,
`positive_points_csv`, and `negative_points_csv` -- nothing under
`src/` or `configs/` is written to. Output goes to
`OUTPUTS_DIR/paper_figures/` (the same directory notebook `09` uses),
so it lands alongside the other paper figures. Row 2's OSM tiles are
fetched live (needs internet access, e.g. on Colab) and cached by
Cartopy for the session.

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running on Colab -- skipping drive mount (paths.yaml must already resolve locally).")

In [ ]:
!pip install -q pandas numpy matplotlib seaborn pyyaml geopandas shapely cartopy

## 1. Paths and output directory

In [ ]:
import yaml
from pathlib import Path

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)

CITIES = paths_cfg["cities"]
OUTPUTS_DIR = Path(paths_cfg["outputs_dir"])
FIG_DIR = OUTPUTS_DIR / "paper_figures"   # same dir notebook 09 uses
FIG_DIR.mkdir(parents=True, exist_ok=True)

CITY_DISPLAY_NAME = {
    "bogor": "Bogor, Indonesia",
    "warsaw": "Warsaw, Poland",
    "krakow": "Krak\u00f3w, Poland",
    "somerville": "Somerville, USA",
}

print("Cities:", CITIES)
print("FIG_DIR:", FIG_DIR)

## 2. Load per-city boundary + positive/negative points

Each city's `boundary_geojson` gives the actual study-area polygon (used both to compute the city's map-marker position -- its centroid -- and to draw the inset outline). Points are loaded straight from `positive_points_csv`/`negative_points_csv`, no filtering beyond what those files already contain.

In [ ]:
import geopandas as gpd
import pandas as pd

city_data = {}
for city in CITIES:
    pc = paths_cfg["per_city"][city]
    entry = {}

    boundary_path = Path(pc["boundary_geojson"])
    if boundary_path.exists():
        gdf = gpd.read_file(boundary_path)
        entry["boundary"] = gdf.geometry.unary_union
        c = entry["boundary"].centroid
        entry["center"] = (c.x, c.y)  # (lon, lat)
    else:
        print(f"  [skip] {boundary_path} not found for {city}.")
        entry["boundary"] = None
        entry["center"] = None

    pos_path, neg_path = Path(pc["positive_points_csv"]), Path(pc["negative_points_csv"])
    entry["positive"] = pd.read_csv(pos_path) if pos_path.exists() else None
    entry["negative"] = pd.read_csv(neg_path) if neg_path.exists() else None
    if entry["positive"] is None:
        print(f"  [skip] {pos_path} not found for {city}.")
    if entry["negative"] is None:
        print(f"  [skip] {neg_path} not found for {city}.")

    city_data[city] = entry
    n_pos = len(entry["positive"]) if entry["positive"] is not None else 0
    n_neg = len(entry["negative"]) if entry["negative"] is not None else 0
    center = entry["center"]
    print(f"{CITY_DISPLAY_NAME.get(city, city)}: center={center}, "
          f"n_positive={n_pos}, n_negative={n_neg}")

## 3. House figure style (inline, self-contained, same as notebook 09)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns


class _PaperStyle:
    FULL_W = 7.0
    FS_TICK, FS_LABEL, FS_TITLE, FS_LETTER, FS_LEGEND = 6.5, 7.5, 8.0, 9.5, 6.5
    MUTED = ["#9fd4c0", "#c3b49a", "#8a7358", "#9aa4cd", "#4a4a73",
             "#8ecae0", "#f2a58c", "#3f8f7d"]
    FOCAL = "#8c2f2f"
    LW_SPINE = 0.7

    def apply(self, font="Liberation Sans"):
        sns.set_theme(style="ticks")
        mpl.rcParams.update({
            "font.family": "sans-serif",
            "font.sans-serif": [font, "Arial", "Helvetica", "DejaVu Sans"],
            "font.size": self.FS_TICK,
            "axes.labelsize": self.FS_LABEL,
            "axes.titlesize": self.FS_TITLE,
            "xtick.labelsize": self.FS_TICK,
            "ytick.labelsize": self.FS_TICK,
            "legend.fontsize": self.FS_LEGEND,
            "axes.linewidth": self.LW_SPINE,
            "axes.grid": False,
            "axes.facecolor": "white",
            "figure.facecolor": "white",
            "legend.frameon": False,
            "savefig.dpi": 300,
            "savefig.bbox": "tight",
            "savefig.pad_inches": 0.02,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        })

    def finish(self, ax, trim=True):
        """Despine + trim for a PLAIN (non-geographic) axes only -- never
        call this on a Cartopy GeoAxes, which manages its own outline/spine
        system and isn't compatible with seaborn's despine."""
        sns.despine(ax=ax, top=True, right=True, trim=trim)

    def panel_letter(self, fig, ax, letter, dx=-0.06, dy=1.08):
        ax.text(dx, dy, letter, transform=ax.transAxes,
                fontsize=self.FS_LETTER, fontweight="bold", ha="left", va="bottom")

    def sparse_yticks(self, ax, n=4):
        ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=n, prune=None))

    def save(self, fig, name):
        fig.savefig(FIG_DIR / f"{name}.pdf")
        fig.savefig(FIG_DIR / f"{name}.png")
        print(f"  [saved] {name}.pdf + {name}.png")


ps = _PaperStyle()
ps.apply()

## 4. World map (row 1) + per-city OSM panels (row 2)

Row 1 is one world map with all four cities pinpointed, each city
coded by both **colour and marker shape** (so identity survives
greyscale printing or colour-vision deficiency, not colour alone).
Row 2 lays the four cities out side by side on a real OpenStreetMap
basemap each, with the study-area boundary, positive/negative points,
and longitude/latitude tick labels on both axes -- panel letters
(a) for the world map, (b)-(e) for the cities in `CITIES` order.

In [ ]:
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.img_tiles as cimgt
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.lines import Line2D

POS_COLOR, NEG_COLOR = ps.MUTED[4], ps.FOCAL  # same pairing as F1
CITY_COLOR = {c: ps.MUTED[i % len(ps.MUTED)] for i, c in enumerate(CITIES)}
CITY_MARKER = {c: m for c, m in zip(CITIES, ["o", "s", "^", "D", "P", "X"])}
OSM_ZOOM = 13  # city-block detail; raise for a tighter study area, lower if panels look pixelated


def plot_city_panel(ax, city, entry, letter):
    boundary, pos_df, neg_df = entry["boundary"], entry["positive"], entry["negative"]
    minx, miny, maxx, maxy = boundary.bounds
    pad_x = max((maxx - minx) * 0.15, 0.01)
    pad_y = max((maxy - miny) * 0.15, 0.01)
    extent = [minx - pad_x, maxx + pad_x, miny - pad_y, maxy + pad_y]
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_image(cimgt.OSM(), OSM_ZOOM, interpolation="bilinear")

    parts = [boundary] if boundary.geom_type == "Polygon" else list(boundary.geoms)
    for part in parts:
        bx, by = part.exterior.xy
        ax.plot(bx, by, color="#1a1a1a", lw=1.2, transform=ccrs.PlateCarree(), zorder=3)

    if neg_df is not None and {"lon", "lat"}.issubset(neg_df.columns):
        ax.scatter(neg_df["lon"], neg_df["lat"], s=5, color=NEG_COLOR, alpha=0.8,
                   edgecolor="none", transform=ccrs.PlateCarree(), zorder=4, label="negative")
    if pos_df is not None and {"lon", "lat"}.issubset(pos_df.columns):
        ax.scatter(pos_df["lon"], pos_df["lat"], s=5, color=POS_COLOR, alpha=0.8,
                   edgecolor="none", transform=ccrs.PlateCarree(), zorder=5, label="positive")

    ax.set_xticks(np.linspace(extent[0], extent[1], 3), crs=ccrs.PlateCarree())
    ax.set_yticks(np.linspace(extent[2], extent[3], 3), crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter(number_format=".2f"))
    ax.yaxis.set_major_formatter(LatitudeFormatter(number_format=".2f"))
    ax.tick_params(labelsize=ps.FS_TICK, length=2.5)

    n_pos = len(pos_df) if pos_df is not None else 0
    n_neg = len(neg_df) if neg_df is not None else 0
    ax.set_title(f"{CITY_DISPLAY_NAME.get(city, city)}\nn={n_pos} positive, {n_neg} negative",
                fontsize=ps.FS_LABEL, fontweight="bold")
    if letter:
        ps.panel_letter(fig, ax, letter)


fig = plt.figure(figsize=(ps.FULL_W * 1.8, ps.FULL_W * 1.05))
gs = fig.add_gridspec(2, len(CITIES), height_ratios=[1.15, 1.0], hspace=0.60, wspace=0.32)

# --- Row 1: world map, one panel spanning every column ---
ax_world = fig.add_subplot(gs[0, :], projection=ccrs.PlateCarree())
ax_world.set_global()
ax_world.add_feature(cfeature.LAND, facecolor="#eee8dd", zorder=0)
ax_world.add_feature(cfeature.OCEAN, facecolor="#dbe7ef", zorder=0)
ax_world.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="0.4", zorder=1)
ax_world.add_feature(cfeature.BORDERS, linewidth=0.25, edgecolor="0.6", zorder=1)
ax_world.set_xticks(np.arange(-180, 181, 60), crs=ccrs.PlateCarree())
ax_world.set_yticks(np.arange(-90, 91, 30), crs=ccrs.PlateCarree())
ax_world.xaxis.set_major_formatter(LongitudeFormatter())
ax_world.yaxis.set_major_formatter(LatitudeFormatter())
ax_world.tick_params(labelsize=ps.FS_TICK, length=2.5)
ax_world.set_title("Study area cities", fontsize=ps.FS_TITLE, fontweight="bold")
ps.panel_letter(fig, ax_world, "a")

world_legend = []
for city in CITIES:
    entry = city_data[city]
    if entry["center"] is None:
        print(f"  [skip] {city} -- no boundary/center available for map placement.")
        continue
    lon, lat = entry["center"]
    ax_world.scatter([lon], [lat], s=45, color=CITY_COLOR[city], marker=CITY_MARKER[city],
                     edgecolor="white", linewidth=0.9, zorder=5, transform=ccrs.PlateCarree())
    world_legend.append(Line2D([], [], marker=CITY_MARKER[city], color="none",
                               markerfacecolor=CITY_COLOR[city], markeredgecolor="white",
                               markersize=7, label=CITY_DISPLAY_NAME.get(city, city)))
ax_world.legend(handles=world_legend, loc="lower left", fontsize=ps.FS_LEGEND,
               frameon=False, handletextpad=0.6, ncol=2)

# --- Row 2: City A | City B | City C | City D, each its own OSM panel ---
panel_letters = "bcdefg"
for i, city in enumerate(CITIES):
    entry = city_data[city]
    if entry["boundary"] is None:
        print(f"  [skip] {city} -- no boundary polygon, panel skipped.")
        continue
    ax_city = fig.add_subplot(gs[1, i], projection=ccrs.PlateCarree())
    plot_city_panel(ax_city, city, entry, panel_letters[i] if i < len(panel_letters) else "")

point_legend = [
    Line2D([], [], marker="o", color="none", markerfacecolor=POS_COLOR, markersize=6, label="positive (crash)"),
    Line2D([], [], marker="o", color="none", markerfacecolor=NEG_COLOR, markersize=6, label="negative (generated)"),
]
fig.legend(handles=point_legend, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02),
          fontsize=ps.FS_LEGEND, frameon=False)

ps.save(fig, "F0_study_area_overview")
plt.show()

In [ ]:
print("Figures in", FIG_DIR, ":")
for p in sorted(FIG_DIR.glob("F0_study_area_overview*")):
    print(" -", p.name)